In [1]:
!pip install -q pyngrok

In [ ]:
import subprocess

print("Installing zstd dependency...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)

print("Installing Ollama engine...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

In [ ]:
import os
import time
import requests
import subprocess
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
except Exception as e:
    print("❌ ERROR: Could not find 'NGROK_AUTH_TOKEN' in Kaggle Secrets.")
    raise e

print("Opening ngrok tunnel...")
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(11434, host_header="localhost").public_url

# Environment settings optimized for high-depth thinking & parallel queries
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_MAX_QUEUE"] = "10" 
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"
os.environ["OLLAMA_HOST"] = "0.0.0.0" 
os.environ["OLLAMA_CONTEXT_LENGTH"] = "65536"

print("\nStarting multi-model Ollama server...")
server_process = subprocess.Popen("ollama serve", shell=True, env=os.environ)
time.sleep(5) 

print("-" * 65)
print("Initiating parallel downloads for all models...")

# Popen starts the downloads simultaneously in the background
pull_1 = subprocess.Popen("ollama pull qwen3.6:27b", shell=True)
pull_2 = subprocess.Popen("ollama pull qwen3.6:latest", shell=True)
pull_3 = subprocess.Popen("ollama pull qwen3.8:27b", shell=True)

# wait() pauses the script until all background downloads finish
pull_1.wait()
pull_2.wait()
pull_3.wait()

print("\n✅ All models downloaded and verified!")
print("\n🔥 Warming up Qwen3.8 into GPU VRAM in xhigh thinking mode...")
warmup_models = ["qwen3.8:27b"]
for model in warmup_models:
    try:
        res = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": "Initialize reasoning memory.",
                "keep_alive": "60m",
                "options": {
                    "num_ctx": 65536,
                    "num_predict": 32768,
                    "temperature": 0.6
                }
            },
            timeout=180
        )
        if res.status_code == 200:
            print(f"✅ Successfully pre-loaded {model} into GPU VRAM!")
            break
    except Exception as e:
        print(f"Warmup status for {model}: {e}")

print("\n" + "="*65)
print("✅ MULTI-MODEL OLLAMA SERVER IS ONLINE WITH QWEN 3.8 WARMED UP!")
print(f"🚀 Base URL for your coding agent:")
print(f"🔗 {public_url}/v1")
print("="*65 + "\n")

try:
    print("Server is running. Do not close this browser tab!")
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down server and tunnel...")
    server_process.terminate()
    ngrok.kill()
